<a href="https://colab.research.google.com/github/jaylink-coder/miss-yaya/blob/main/yaya-ai/notebooks/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Yaya-125M Training — Google Colab

Trains the Yaya-125M model (~129M parameters) on OpenWebText.

**Requirements:** Runtime → Change runtime type → **T4 GPU**

**Estimated time:** ~10–12 hours for 100k steps (full Colab session)

## 1. Check GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('PyTorch:', torch.__version__)

## 2. Mount Google Drive (for persistent checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/yaya-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', CHECKPOINT_DIR)

## 3. Clone Yaya repo

In [ ]:
REPO_URL = 'https://github.com/jaylink-coder/miss-yaya.git'

!git clone {REPO_URL} /content/miss-yaya
%cd /content/miss-yaya

## 4. Install dependencies

In [ ]:
!pip install -q sentencepiece datasets wandb
print('Dependencies installed.')

## 5. Download & prepare training data (OpenWebText, ~38GB — use a sample)

In [ ]:
import os, sys
sys.path.insert(0, '/content/yaya-ai')

from datasets import load_dataset
import numpy as np

# Load a 10% sample of OpenWebText (~700MB of text, ~150M tokens)
# Enough for a meaningful Colab run. Remove split= for the full dataset.
print('Downloading OpenWebText sample...')
ds = load_dataset('openwebtext', split='train[:10%]', trust_remote_code=True)
print(f'Loaded {len(ds):,} documents')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openwebtext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'openwebtext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

...

plain_text/train-00076-of-00080.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

plain_text/train-00077-of-00080.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

plain_text/train-00078-of-00080.parquet:   0%|          | 0.00/301M [00:00<?, ?B/s]

plain_text/train-00079-of-00080.parquet:   0%|          | 0.00/303M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8013769 [00:00<?, ? examples/s]

In [ ]:
from src.tokenizer.tokenizer import YayaTokenizer

TOKENIZER_PATH = 'data/tokenizer/yaya_tokenizer.model'
tokenizer = YayaTokenizer(TOKENIZER_PATH)
print('Tokenizer vocab size:', tokenizer.vocab_size)

# Tokenize and write shards
TRAIN_DIR = 'data/processed/openwebtext/train'
EVAL_DIR  = 'data/processed/openwebtext/eval'
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)

# Split: 99% train, 1% eval
split = ds.train_test_split(test_size=0.01, seed=42)
train_ds = split['train']
eval_ds  = split['test']

def tokenize_and_save(dataset, out_path, filename):
    all_tokens = []
    for i, row in enumerate(dataset):
        tokens = tokenizer.encode(row['text'])
        all_tokens.extend(tokens)
        if (i + 1) % 10000 == 0:
            print(f'  {i+1:,} docs, {len(all_tokens):,} tokens')
    arr = np.array(all_tokens, dtype=np.uint16)
    arr.tofile(os.path.join(out_path, filename))
    print(f'Saved {len(arr):,} tokens to {out_path}/{filename}')
    return len(arr)

print('Tokenizing train split...')
n_train = tokenize_and_save(train_ds, TRAIN_DIR, 'shard_00000.bin')

print('Tokenizing eval split...')
n_eval = tokenize_and_save(eval_ds, EVAL_DIR, 'eval.bin')

print(f'\nTotal: {n_train:,} train tokens, {n_eval:,} eval tokens')

## 6. Update config to save checkpoints to Drive

In [ ]:
import yaml

with open('configs/training/train_125m.yaml') as f:
    train_cfg = yaml.safe_load(f)

# Point checkpoints to Drive so they survive session resets
train_cfg['checkpointing']['save_dir'] = CHECKPOINT_DIR

with open('configs/training/train_125m.yaml', 'w') as f:
    yaml.dump(train_cfg, f)

print('Checkpoint dir set to:', CHECKPOINT_DIR)

## 7. (Optional) Login to W&B for live loss curves
Skip this cell if you don't want to track metrics.

In [ ]:
import wandb
wandb.login()  # paste your API key when prompted
print('W&B ready.')

## 8. Train!

In [ ]:
# Check for existing checkpoint to resume from
import glob
ckpts = sorted(glob.glob(f'{CHECKPOINT_DIR}/checkpoint-*'))
resume_flag = f'--resume {ckpts[-1]}' if ckpts else ''
print('Resuming from:', ckpts[-1] if ckpts else 'scratch')

!python scripts/train.py \
    --model_config configs/model/yaya_125m.yaml \
    --train_config configs/training/train_125m.yaml \
    {resume_flag}

## 9. Quick generation test
Run after training (or after loading a checkpoint).

In [ ]:
import torch, glob
from src.utils.config import load_model_config
from src.model.yaya_model import YayaForCausalLM
from src.tokenizer.tokenizer import YayaTokenizer
from src.inference.generator import TextGenerator
from src.training.checkpointing import CheckpointManager

model_config = load_model_config('configs/model/yaya_125m.yaml')
model = YayaForCausalLM(model_config)

# Load latest checkpoint
ckpt_mgr = CheckpointManager(save_dir=CHECKPOINT_DIR)
ckpt_mgr.load(model)

model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

tokenizer = YayaTokenizer('data/tokenizer/yaya_tokenizer.model')
generator = TextGenerator(model, tokenizer, device=device)

prompt = 'The history of artificial intelligence'
output = generator.generate(prompt, max_new_tokens=100, temperature=0.8, top_p=0.9)
print(output)